# EventLens Stage A · Quant & Reward Lab (Blueprint v2)

**Interactive Empirical Research & Model Benchmark Notebook**

Aligned with [Commander Blueprint v2](../docs/blueprint-v2.md) ([中文版](../docs/blueprint-v2_CN.md)):
- **Dataset:** 14+ days of real cloud Pepperstone Demo EUR/USD Bid/Ask ticks (`M1/M5/M15/H1`) + arrival-clocked Tier-1 English macro news (`investingLive`, `ECB`, `Fed`, `US BEA`, `Forex Factory`) + real executed Demo trades.
- **Zero-Friction Sync:** Auto-fetches sanitized `chart-lab.json` from GitHub `dashboard-live` (or reads local `export-research` bundle) + optionally merges Boss manual K-line labels (`chart-lab-labels.json`).
- **Bidirectional News Attribution:** Separates **post-arrival causal impact (`0-15m`, `15-60m`)** from **pre-arrival front-running / feed lag (`-60m` to `0m`, highlighting `-5m`)** across the 4 FX liquidity sessions.
- **Standardized Net-R Evaluation:** Evaluates 4 realistic exit templates (`T1_scalp`, `T2_mid_swing`, `T3_large_swing`, `T4_dynamic_atr`) minus real spread + Razor commission (`0.60p` round-trip) with near-stop MAE penalties & miss/avoid shaping.
- **Out-of-Sample Arena:** Chronological `70%` discovery / `30%` blind OOS split benchmarking **Gemini vs. DeepSeek** head-to-head.

In [ ]:
# 1. Environment & EventLens Research Module Setup (works seamlessly in Colab or local repo)
import importlib.util
import json
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if importlib.util.find_spec("eventlens") is None:
    if Path("src/eventlens/research_lab.py").is_file():
        sys.path.insert(0, str(Path("src").resolve()))
    else:
        import subprocess

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "git+https://github.com/lohis-aimbot/eventlens.git@main",
            ]
        )

from eventlens.research_lab import (
    score_decision_net_r,
    simulate_exit_templates,
    split_chronological_oos,
)

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 140)
print("EventLens Stage A Quant & Reward Lab ready.")

In [ ]:
# 2. Load Sanitized 14+ Day Cloud Dataset + Optional Boss Manual Chart Lab Annotations
REMOTE_CHART_LAB_URL = (
    "https://raw.githubusercontent.com/lohis-aimbot/eventlens/dashboard-live/chart-lab.json"
)
LOCAL_BUNDLE_DIR = Path("research-bundle")  # Optional output of `eventlens-shadow export-research`
BOSS_LABELS_FILE = Path("chart-lab-labels.json")  # Optional export from Web Dashboard -> Chart Lab

if (LOCAL_BUNDLE_DIR / "chart-lab.json").is_file():
    chart_lab = json.loads((LOCAL_BUNDLE_DIR / "chart-lab.json").read_text(encoding="utf-8"))
    data_source = str(LOCAL_BUNDLE_DIR / "chart-lab.json")
else:
    with urllib.request.urlopen(REMOTE_CHART_LAB_URL, timeout=30) as resp:
        chart_lab = json.loads(resp.read().decode("utf-8"))
    data_source = REMOTE_CHART_LAB_URL

df_m5 = pd.DataFrame(chart_lab.get("bars", {}).get("m5", []))
df_m15 = pd.DataFrame(chart_lab.get("bars", {}).get("m15", []))
df_h1 = pd.DataFrame(chart_lab.get("bars", {}).get("h1", []))
df_events = pd.DataFrame(chart_lab.get("events", []))
df_markers = pd.DataFrame(chart_lab.get("algo_markers", []))
df_trades = pd.DataFrame(chart_lab.get("trades", []))

# If local full M1 bars exist in research-bundle, load them for exact 1-minute counterfactual simulation.
if (LOCAL_BUNDLE_DIR / "bars_m1.csv.gz").is_file():
    df_m1 = pd.read_csv(LOCAL_BUNDLE_DIR / "bars_m1.csv.gz")
    sim_bars = df_m1.to_dict(orient="records")
else:
    # Fallback proxy using M5 Bid/Ask approximation (mid +/- half spread)
    sim_bars = []
    for b in chart_lab.get("bars", {}).get("m5", []):
        half_sp = (float(b.get("sa", 0.2)) * 0.0001) * 0.5
        sim_bars.append(
            {
                "t": int(b["t"]),
                "iso": b["iso"],
                "o": b["o"],
                "h": b["h"],
                "l": b["l"],
                "c": b["c"],
                "bo": b["o"] - half_sp,
                "bh": b["h"] - half_sp,
                "bl": b["l"] - half_sp,
                "bc": b["c"] - half_sp,
                "ao": b["o"] + half_sp,
                "ah": b["h"] + half_sp,
                "al": b["l"] + half_sp,
                "ac": b["c"] + half_sp,
                "sa": b.get("sa", 0.2),
                "sm": b.get("sm", 0.5),
                "n": b.get("n", 1),
                "s": b.get("s", "london"),
                "atr14": b.get("atr14", 10.0),
            }
        )

boss_labels = {}
if BOSS_LABELS_FILE.is_file():
    raw_lbls = json.loads(BOSS_LABELS_FILE.read_text(encoding="utf-8"))
    boss_labels = raw_lbls.get("labels", raw_lbls) if isinstance(raw_lbls, dict) else {}

win = chart_lab.get("window", {})
print(f"Source: {data_source}")
print(
    f"Window: {win.get('first_iso')} -> {win.get('last_iso')} | Ticks: {win.get('total_ticks'):,}"
)
print(
    f"M5={len(df_m5)} | M15={len(df_m15)} | H1={len(df_h1)} | Events={len(df_events)} | AlgoMarkers={len(df_markers)} | Trades={len(df_trades)} | BossLabels={len(boss_labels)}"
)

## 3. FX Session Spread, Volatility & 3-Tier Optimal Opportunities

Inspect liquidity and spread quality across the 4 sessions (`asian`, `london`, `overlap`, `late_ny`) alongside causal 3-tier `MFE/MAE` opportunities (`T1 >= 10p`, `T2 >= 15p`, `T3 >= 25p`).

In [ ]:
sess_df = pd.DataFrame(chart_lab.get("session_stats", {})).T
display(sess_df)

if not df_markers.empty:
    marker_summary = (
        df_markers.groupby(["session", "tier_key", "side"])
        .agg(
            count=("t", "count"),
            avg_mfe_pips=("mfe_pips", "mean"),
            avg_mae_pips=("mae_pips", "mean"),
            median_peak_min=("time_to_peak_min", "median"),
        )
        .round(2)
    )
    display(marker_summary)

if not df_trades.empty:
    cols = [
        c
        for c in [
            "opened_at",
            "session",
            "side",
            "lots",
            "pips",
            "net_pnl_usd",
            "mfe_pips",
            "mae_pips",
            "drift_1h_pips",
            "drift_4h_pips",
            "exit_reason_en",
        ]
        if c in df_trades.columns
    ]
    print("\n--- Live Pepperstone Demo Trades Autopsy ---")
    display(df_trades[cols])

## 4. Bidirectional News Attribution (`-60m` Front-Running / Feed-Lag vs. `0..+60m` Causal Moves)

Tests each news provider for:
- **Pre-arrival drift (`pre_5m_pips`, `pre_15m_pips`, `pre_60m_pips`):** Flags headlines where price moved sharply before local arrival (`front_run_flag`).
- **Post-arrival causal reaction (`post_15m_pips`, `post_15m_to_60m_pips`, `post_60m_pips`, `post_4h_pips`).**

In [ ]:
prov_df = pd.DataFrame(chart_lab.get("provider_stats", {})).T
display(prov_df)

if not df_events.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_events["abs_pre_5m"] = df_events["pre_5m_pips"].abs()
    df_events["abs_post_15m"] = df_events["post_15m_pips"].abs()
    df_events["abs_post_60m"] = df_events["post_60m_pips"].abs()
    by_sess = df_events.groupby("session")[["abs_pre_5m", "abs_post_15m", "abs_post_60m"]].mean()
    by_sess.plot(
        kind="bar", ax=axes[0], title="Mean |Move| (pips) Around News Arrival by FX Session"
    )
    axes[0].set_ylabel("Pips")

    top_movers = (
        df_events.dropna(subset=["post_15m_pips"])
        .assign(abs15=lambda d: d["post_15m_pips"].abs())
        .sort_values("abs15", ascending=False)
        .head(15)
    )
    axes[1].scatter(
        top_movers["pre_5m_pips"].fillna(0), top_movers["post_15m_pips"].fillna(0), alpha=0.8
    )
    axes[1].axhline(0, color="grey", lw=0.8)
    axes[1].axvline(0, color="grey", lw=0.8)
    axes[1].set_title("Top-15 High-Impact Events: Pre-5m vs Post-15m (pips)")
    axes[1].set_xlabel("Pre-arrival -5m (pips)")
    axes[1].set_ylabel("Post-arrival +15m (pips)")
    plt.tight_layout()
    plt.show()

    front_runs = df_events[df_events["front_run_flag"].astype(bool)].head(10)
    print(
        f"Flagged Front-Run / Lagged Wires: {int(df_events['front_run_flag'].astype(bool).sum())} / {len(df_events)}"
    )
    display(
        front_runs[
            ["iso", "session", "provider", "pre_5m_pips", "post_15m_pips", "runtime_state", "title"]
        ]
    )

## 5. Exit Template Matrix (`T1..T4`) + Natural Drift + Standardized `Net-R` Evaluation

Simulates every optimal marker, runtime signal, and Boss manual label across **T1 (`10/15p 90m`)**, **T2 (`14/24p+trail 4h`)**, **T3 (`18/36p 8h`)**, and **T4 (`ATR-scaled`)** after Bid/Ask spread + `0.60p` Razor commission.

In [ ]:
m1_ts = [int(b["t"]) for b in sim_bars]


def evaluate_signal_table(signals):
    rows = []
    for sig in signals:
        ts = int(sig["entry_t"] if "entry_t" in sig else sig["t"])
        side = sig["side"]
        atr = float(sig.get("atr14") or 10.0)
        sim = simulate_exit_templates(sim_bars, ts, side, atr14_pips=atr, m1_ts=m1_ts)
        if not sim.get("available"):
            continue
        rec = {
            "t": ts,
            "iso": sig.get("iso"),
            "session": sig.get("session"),
            "side": side,
            "source": sig.get("source", "algo"),
            "drift_30m": sim["drift_30m_pips"],
            "drift_1h": sim["drift_1h_pips"],
            "drift_4h": sim["drift_4h_pips"],
        }
        for tname, res in sim["templates"].items():
            rec[f"{tname}_r"] = res.get("r_net")
            rec[f"{tname}_pips"] = res.get("net_pips")
            sc = score_decision_net_r(
                action="SET_TARGET",
                side=side,
                long_eval=res if side == "long" else None,
                short_eval=res if side == "short" else None,
                has_qualified_setup=True,
            )
            rec[f"{tname}_score"] = sc["score"]
        rows.append(rec)
    return pd.DataFrame(rows)


algo_eval_df = evaluate_signal_table(chart_lab.get("algo_markers", []))
if not algo_eval_df.empty:
    summary_cols = [
        c for c in algo_eval_df.columns if c.endswith(("_r", "_score")) or c.startswith("drift_")
    ]
    print("--- 3-Tier Optimal Entry Performance Across 4 Exit Templates ---")
    display(algo_eval_df[summary_cols].agg(["count", "mean", "sum"]).round(3))

## 6. Chronological `70% / 30%` Out-of-Sample Split & **Gemini vs. DeepSeek Arena**

Evaluates candidate macro-decision models strictly on the **unseen final 30%** chronological window.
- Set `RUN_LIVE_API_ARENA = True` in Colab (with `DEEPSEEK_API_KEY` and `GEMINI_API_KEY` in Colab Secrets / env vars) to run real head-to-head model inference on candidate news windows.
- Keep `RUN_LIVE_API_ARENA = False` (default) to verify the deterministic scoring harness and baseline benchmark cleanly with zero API cost.

In [ ]:
RUN_LIVE_API_ARENA = False
ARENA_MAX_EVENTS = 25
ARENA_TEMPLATE = "T2_mid_swing"
GEMINI_MODEL = "gemini-2.5-pro"
DEEPSEEK_MODEL = "deepseek-reasoner"


def _get_secret(name: str) -> str:
    import os

    val = os.environ.get(name, "").strip()
    if val:
        return val
    try:
        from google.colab import userdata

        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


def build_arena_prompt(ev):
    return json.dumps(
        {
            "task": "Evaluate causal EUR/USD intraday reaction on news arrival. Reply strictly in JSON.",
            "output_schema": {
                "action": "SET_TARGET | HOLD | IGNORE_EVENT",
                "side": "long | short | null",
                "rationale": "short factual explanation",
            },
            "event": {
                "arrival_utc": ev["iso"],
                "fx_session": ev["session"],
                "provider": ev["provider"],
                "headline": ev["title"],
                "snippet": ev["snippet"],
                "pre_60m_pips": ev["pre_60m_pips"],
                "pre_15m_pips": ev["pre_15m_pips"],
                "pre_5m_pips": ev["pre_5m_pips"],
            },
        },
        ensure_ascii=False,
    )


def call_gemini_json(prompt_text: str, api_key: str, model: str = GEMINI_MODEL) -> dict:
    import httpx

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    body = {
        "contents": [{"parts": [{"text": prompt_text}]}],
        "generationConfig": {"responseMimeType": "application/json", "temperature": 0.1},
    }
    r = httpx.post(url, params={"key": api_key}, json=body, timeout=45.0)
    r.raise_for_status()
    txt = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(txt)


def call_deepseek_json(prompt_text: str, api_key: str, model: str = DEEPSEEK_MODEL) -> dict:
    import httpx

    r = httpx.post(
        "https://api.deepseek.com/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt_text}],
            "response_format": {"type": "json_object"},
        },
        timeout=60.0,
    )
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"]
    return json.loads(txt)


candidate_events = [
    e
    for e in chart_lab.get("events", [])
    if e.get("runtime_state") not in ("filtered_out",) and e.get("post_60m_pips") is not None
]
in_sample_events, oos_events, oos_split_ts = split_chronological_oos(
    candidate_events, ts_key="t", train_ratio=0.70
)
print(
    f"Chronological Split: In-Sample={len(in_sample_events)} events | Unseen Blind OOS={len(oos_events)} events"
)

marker_ts_list = [int(m["t"]) for m in chart_lab.get("algo_markers", [])]


def score_model_predictions(preds, template_name=ARENA_TEMPLATE):
    records = []
    for ev, pred in preds:
        ts = int(ev["t"])
        long_eval = (
            simulate_exit_templates(sim_bars, ts, "long", m1_ts=m1_ts).get("templates") or {}
        ).get(template_name)
        short_eval = (
            simulate_exit_templates(sim_bars, ts, "short", m1_ts=m1_ts).get("templates") or {}
        ).get(template_name)
        has_setup = any(abs(mt - ts) <= 900 for mt in marker_ts_list)
        sc = score_decision_net_r(
            action=str(pred.get("action", "HOLD")),
            side=pred.get("side"),
            long_eval=long_eval,
            short_eval=short_eval,
            has_qualified_setup=has_setup,
        )
        records.append(
            {
                "t": ts,
                "iso": ev["iso"],
                "session": ev["session"],
                "action": pred.get("action"),
                **sc,
            }
        )
    return pd.DataFrame(records)


model_runs = [
    ("Baseline_Always_Flat", [(ev, {"action": "HOLD", "side": None}) for ev in oos_events]),
    (
        "Recorded_VM_DeepSeek",
        [
            (
                ev,
                {
                    "action": ev.get("portfolio_action") or "HOLD",
                    "side": ev.get("target_direction"),
                },
            )
            for ev in oos_events
        ],
    ),
]

if RUN_LIVE_API_ARENA and oos_events:
    arena_slice = oos_events[:ARENA_MAX_EVENTS]
    for label, key_name, caller_fn in [
        (f"Arena_{GEMINI_MODEL}", "GEMINI_API_KEY", call_gemini_json),
        (f"Arena_{DEEPSEEK_MODEL}", "DEEPSEEK_API_KEY", call_deepseek_json),
    ]:
        secret = _get_secret(key_name)
        if not secret:
            print(f"Skipping {label}: {key_name} not configured in Colab Secrets / env.")
            continue
        preds = []
        for ev in arena_slice:
            try:
                preds.append((ev, caller_fn(build_arena_prompt(ev), secret)))
            except Exception as exc:
                preds.append((ev, {"action": "HOLD", "side": None, "error": str(exc)}))
        model_runs.append((label, preds))

arena_summary = []
for name, preds in model_runs:
    df_res = score_model_predictions(preds)
    if df_res.empty:
        continue
    entries_df = df_res[df_res["regime"] == "active_entry"]
    win_rate = (
        round(float((entries_df["r_net"] > 0).mean() * 100.0), 1) if not entries_df.empty else None
    )
    mean_score = round(float(df_res["score"].dropna().mean()), 4)
    total_r = round(float(df_res["r_net"].dropna().sum()), 3)
    gate_pass = bool(mean_score > 0 and total_r > 0 and len(entries_df) >= 3)
    arena_summary.append(
        {
            "model": name,
            "events": len(df_res),
            "entries": int(len(entries_df)),
            "entry_win_rate_pct": win_rate,
            "total_score": round(float(df_res["score"].dropna().sum()), 3),
            "mean_score": mean_score,
            "total_r_net": total_r,
            "go_live_gate_pass": gate_pass,
        }
    )

arena_df = pd.DataFrame(arena_summary)
print(f"\n--- Stage A Blind OOS Arena & Go-Live Gate ({ARENA_TEMPLATE}) ---")
display(arena_df)

# Export clean Stage A Case Memory artifact (golden wins, traps & Boss labels)
case_memory = {
    "schema_version": "stage-a-case-memory-v1",
    "window": chart_lab.get("window"),
    "top_algo_setups": chart_lab.get("algo_markers", [])[:40],
    "boss_labels": boss_labels,
    "demo_trades_autopsy": chart_lab.get("trades", []),
}
Path("stage_a_case_memory.json").write_text(
    json.dumps(case_memory, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved stage_a_case_memory.json successfully.")